# Arabic Heritage ABSA Transformer Experiment

**Project:** Aspect-Based Sentiment Analysis of Arabic Reviews for Saudi Heritage Sites Using Transformer Models

This notebook trains and compares **AraBERT, MARBERT, and QARiB** on the heritage subset prepared from the uploaded project dataset.

### Fixed experimental design
- 1,084 heritage-category rows in the original file.
- 1 row has no aspect labels, so **1,083 labeled reviews** are used for modeling.
- Six aspects: price, cleanliness, facilities, service & staff, environment, overall experience.
- Labels: **-1 = Negative, 0 = Neutral, 1 = Positive**. Blank cells mean *aspect not annotated* and are masked from loss/evaluation.
- Fixed **80/20 review-level split** from `heritage_modeling_data.csv`: 866 train reviews, 217 test reviews.
- Architecture: one shared transformer encoder with **six 3-class sentiment heads**.
- Same setup for all three backbones for a fair comparison.
- Primary metric: **Macro-F1**.

The setup follows the same general fine-tuning choices used by the published Saudi tourism ABSA benchmark (max length 128, learning rate 2e-5, 3 epochs), while evaluating only the heritage subset.


In [ ]:
# Install the libraries in Google Colab.
!pip -q install transformers==4.57.1 accelerate scikit-learn matplotlib


In [ ]:
# Upload the prepared CSV supplied with this notebook.
from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded.keys()))
print("Using:", DATA_PATH)


In [ ]:
# Imports and reproducibility
import csv, os, random, json, math
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("WARNING: Use a Colab GPU runtime (Runtime > Change runtime type > T4 GPU) before training.")


In [ ]:
# Configuration
ASPECTS = [
    "price",
    "cleanliness",
    "facilities",
    "service_staff",
    "environment",
    "overall_experience",
]

ASPECT_DISPLAY = {
    "price": "Price",
    "cleanliness": "Cleanliness",
    "facilities": "Facilities",
    "service_staff": "Service & Staff",
    "environment": "Environment",
    "overall_experience": "Overall Experience",
}

# Original labels -> classifier IDs
LABEL_TO_ID = {-1: 0, 0: 1, 1: 2}
ID_TO_LABEL = {0: -1, 1: 0, 2: 1}
LABEL_NAMES = ["Negative", "Neutral", "Positive"]

MODELS = {
    "AraBERT": "aubmindlab/bert-base-arabertv02",
    "MARBERT": "UBC-NLP/MARBERT",
    "QARiB": "ahmedabdelali/bert-base-qarib",
}

MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3
WEIGHT_DECAY = 0.01
DROPOUT = 0.2


In [ ]:
# Load the prepared review-level data.
def parse_label(value):
    value = str(value).strip()
    if value == "":
        return -100  # ignored / missing aspect
    return LABEL_TO_ID[int(float(value))]

rows = []
with open(DATA_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    for r in reader:
        if r["split"] == "Excluded_no_labels":
            continue
        row = {
            "review_id": int(r["review_id"]),
            "split": r["split"],
            "place": r["place"],
            "city": r["city"],
            "text": r["processed_review"],
            "labels": [parse_label(r[a]) for a in ASPECTS],
        }
        rows.append(row)

train_rows = [r for r in rows if r["split"] == "Train"]
test_rows = [r for r in rows if r["split"] == "Test"]

print("Usable reviews:", len(rows))
print("Train:", len(train_rows))
print("Test:", len(test_rows))
assert len(rows) == 1083
assert len(train_rows) == 866
assert len(test_rows) == 217


In [ ]:
# Inspect training/test label support.
def support_table(data_rows):
    result = {}
    for j, aspect in enumerate(ASPECTS):
        c = Counter()
        for r in data_rows:
            y = r["labels"][j]
            if y != -100:
                c[ID_TO_LABEL[y]] += 1
        result[aspect] = c
    return result

print("TRAIN SUPPORT")
for a, c in support_table(train_rows).items():
    print(a, dict(c))

print("\nTEST SUPPORT")
for a, c in support_table(test_rows).items():
    print(a, dict(c))

print("\nImportant: the heritage subset contains only one cleanliness-neutral example in total.")
print("Per-aspect results must therefore be interpreted together with class support.")


In [ ]:
# Dataset and dynamic padding.
class ReviewDataset(Dataset):
    def __init__(self, rows, tokenizer):
        self.rows = rows
        self.enc = tokenizer(
            [r["text"] for r in rows],
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        item = {k: self.enc[k][idx] for k in self.enc}
        item["labels_multi"] = self.rows[idx]["labels"]
        item["review_id"] = self.rows[idx]["review_id"]
        item["place"] = self.rows[idx]["place"]
        return item

def make_collate(tokenizer):
    padder = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    def collate(batch):
        labels = torch.tensor([x.pop("labels_multi") for x in batch], dtype=torch.long)
        review_ids = [x.pop("review_id") for x in batch]
        places = [x.pop("place") for x in batch]
        model_inputs = padder(batch)
        model_inputs["labels_multi"] = labels
        model_inputs["review_ids"] = review_ids
        model_inputs["places"] = places
        return model_inputs
    return collate


In [ ]:
# Compute one class-weight vector per aspect from TRAINING labels only.
def compute_aspect_weights(train_rows):
    weights = []
    for j, aspect in enumerate(ASPECTS):
        counts = Counter(r["labels"][j] for r in train_rows if r["labels"][j] != -100)
        n = sum(counts.values())
        w = []
        for cls in range(3):
            count = counts.get(cls, 0)
            w.append(0.0 if count == 0 else n / (3.0 * count))
        weights.append(torch.tensor(w, dtype=torch.float))
        print(aspect, "counts=", {LABEL_NAMES[k]: counts.get(k,0) for k in range(3)},
              "weights=", [round(x,3) for x in w])
    return weights

ASPECT_WEIGHTS = compute_aspect_weights(train_rows)


In [ ]:
# Multi-head ABSA model: shared encoder + one 3-class head per aspect.
class MultiHeadABSA(nn.Module):
    def __init__(self, model_name, aspect_weights, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleList([nn.Linear(hidden, 3) for _ in ASPECTS])
        self.aspect_weights = [w.clone() for w in aspect_weights]

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels_multi=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**kwargs)
        pooled = out.last_hidden_state[:, 0, :]  # [CLS]
        pooled = self.dropout(pooled)
        logits = [head(pooled) for head in self.heads]

        loss = None
        if labels_multi is not None:
            losses = []
            for j, head_logits in enumerate(logits):
                y = labels_multi[:, j]
                mask = y != -100
                if mask.any():
                    loss_fn = nn.CrossEntropyLoss(
                        weight=self.aspect_weights[j].to(head_logits.device)
                    )
                    losses.append(loss_fn(head_logits[mask], y[mask]))
            loss = torch.stack(losses).mean()
        return loss, logits


In [ ]:
# Evaluation.
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    true_by_aspect = {a: [] for a in ASPECTS}
    pred_by_aspect = {a: [] for a in ASPECTS}
    prediction_rows = []

    for batch in loader:
        labels = batch.pop("labels_multi")
        review_ids = batch.pop("review_ids")
        places = batch.pop("places")
        model_inputs = {k: v.to(device) for k, v in batch.items()}

        _, logits = model(**model_inputs)
        preds = [x.argmax(dim=-1).cpu() for x in logits]

        for b in range(labels.size(0)):
            for j, aspect in enumerate(ASPECTS):
                y = int(labels[b, j].item())
                if y == -100:
                    continue
                p = int(preds[j][b].item())
                true_by_aspect[aspect].append(y)
                pred_by_aspect[aspect].append(p)
                prediction_rows.append({
                    "review_id": review_ids[b],
                    "place": places[b],
                    "aspect": aspect,
                    "true_id": y,
                    "pred_id": p,
                    "true_sentiment": LABEL_NAMES[y],
                    "pred_sentiment": LABEL_NAMES[p],
                })

    # Overall = all labeled aspect-sentiment pairs pooled together.
    y_all = []
    p_all = []
    for a in ASPECTS:
        y_all.extend(true_by_aspect[a])
        p_all.extend(pred_by_aspect[a])

    overall_p, overall_r, overall_f1, _ = precision_recall_fscore_support(
        y_all, p_all, labels=[0,1,2], average="macro", zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_all, p_all),
        "macro_precision": overall_p,
        "macro_recall": overall_r,
        "macro_f1": overall_f1,
        "n": len(y_all),
    }

    per_aspect = []
    for a in ASPECTS:
        yt, yp = true_by_aspect[a], pred_by_aspect[a]
        # Use observed true classes for per-aspect Macro-F1 because cleanliness has
        # only one neutral label in the entire heritage subset, making 3-class
        # holdout evaluation impossible for that aspect.
        observed_labels = sorted(set(yt))
        p, r, f1, _ = precision_recall_fscore_support(
            yt, yp, labels=observed_labels, average="macro", zero_division=0
        )
        support = Counter(yt)
        per_aspect.append({
            "aspect": a,
            "accuracy": accuracy_score(yt, yp),
            "macro_precision": p,
            "macro_recall": r,
            "macro_f1": f1,
            "n": len(yt),
            "negative_support": support.get(0,0),
            "neutral_support": support.get(1,0),
            "positive_support": support.get(2,0),
        })

    return overall, per_aspect, prediction_rows


In [ ]:
# Training function.
def train_one_model(display_name, model_name):
    print("\n" + "="*80)
    print("Training", display_name, "->", model_name)
    print("="*80)

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    train_ds = ReviewDataset(train_rows, tokenizer)
    test_ds = ReviewDataset(test_rows, tokenizer)

    collate = make_collate(tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

    model = MultiHeadABSA(model_name, ASPECT_WEIGHTS, dropout=DROPOUT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(0.1 * total_steps)),
        num_training_steps=total_steps
    )

    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for step, batch in enumerate(train_loader, start=1):
            labels = batch.pop("labels_multi").to(device)
            batch.pop("review_ids")
            batch.pop("places")
            inputs = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad(set_to_none=True)
            loss, _ = model(**inputs, labels_multi=labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += float(loss.item())

        print(f"Epoch {epoch+1}/{EPOCHS} - mean training loss: {running/len(train_loader):.4f}")

    overall, per_aspect, preds = evaluate(model, test_loader)
    print("Overall:", {k: round(v,4) if isinstance(v,float) else v for k,v in overall.items()})
    for r in per_aspect:
        print(r["aspect"], "Macro-F1=", round(r["macro_f1"],4), "N=", r["n"])

    # Free GPU memory after extracting metrics/predictions.
    model.to("cpu")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return overall, per_aspect, preds


In [ ]:
# Run all three transformer experiments.
all_overall = []
all_per_aspect = []
predictions_by_model = {}

for display_name, model_name in MODELS.items():
    overall, per_aspect, preds = train_one_model(display_name, model_name)
    all_overall.append({"model": display_name, **overall})
    for r in per_aspect:
        all_per_aspect.append({"model": display_name, **r})
    predictions_by_model[display_name] = preds

print("\nFinished all models.")


In [ ]:
# Display final overall table.
print("OVERALL RESULTS")
print(f"{'Model':<12} {'Accuracy':>10} {'Macro-P':>10} {'Macro-R':>10} {'Macro-F1':>10}")
for r in sorted(all_overall, key=lambda x: x["macro_f1"], reverse=True):
    print(f"{r['model']:<12} {r['accuracy']:>10.3f} {r['macro_precision']:>10.3f} "
          f"{r['macro_recall']:>10.3f} {r['macro_f1']:>10.3f}")


In [ ]:
# Display per-aspect Macro-F1 table.
print(f"{'Aspect':<22} {'AraBERT':>10} {'MARBERT':>10} {'QARiB':>10}")
lookup = {(r["aspect"], r["model"]): r for r in all_per_aspect}
for a in ASPECTS:
    print(f"{ASPECT_DISPLAY[a]:<22} "
          f"{lookup[(a,'AraBERT')]['macro_f1']:>10.3f} "
          f"{lookup[(a,'MARBERT')]['macro_f1']:>10.3f} "
          f"{lookup[(a,'QARiB')]['macro_f1']:>10.3f}")


In [ ]:
# Save result CSV files for the result tables.
with open("overall_transformer_results.csv", "w", encoding="utf-8-sig", newline="") as f:
    fields = ["model","accuracy","macro_precision","macro_recall","macro_f1","n"]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(all_overall)

with open("per_aspect_transformer_results.csv", "w", encoding="utf-8-sig", newline="") as f:
    fields = ["model","aspect","accuracy","macro_precision","macro_recall","macro_f1","n",
              "negative_support","neutral_support","positive_support"]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(all_per_aspect)

best = max(all_overall, key=lambda x: x["macro_f1"])["model"]
print("Best model by overall Macro-F1:", best)

with open("best_model_test_predictions.csv", "w", encoding="utf-8-sig", newline="") as f:
    fields = ["review_id","place","aspect","true_id","pred_id","true_sentiment","pred_sentiment"]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(predictions_by_model[best])

print("Saved result files.")


In [ ]:
# Figure 1 — Overall Macro-F1 comparison
models = [r["model"] for r in all_overall]
scores = [r["macro_f1"] for r in all_overall]

plt.figure(figsize=(7,4))
plt.bar(models, scores)
plt.ylim(0, 1)
plt.ylabel("Macro-F1")
plt.title("Overall Macro-F1 on Saudi Heritage Review Subset")
for i, v in enumerate(scores):
    plt.text(i, v + 0.015, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.savefig("figure_overall_macro_f1.png", dpi=200)
plt.show()


In [ ]:
# Figure 2 — Per-aspect Macro-F1
x = np.arange(len(ASPECTS))
width = 0.25

plt.figure(figsize=(12,5))
for k, model_name in enumerate(["AraBERT","MARBERT","QARiB"]):
    vals = [lookup[(a,model_name)]["macro_f1"] for a in ASPECTS]
    plt.bar(x + (k-1)*width, vals, width, label=model_name)

plt.xticks(x, [ASPECT_DISPLAY[a] for a in ASPECTS], rotation=25, ha="right")
plt.ylim(0,1)
plt.ylabel("Macro-F1")
plt.title("Macro-F1 by Aspect")
plt.legend()
plt.tight_layout()
plt.savefig("figure_per_aspect_macro_f1.png", dpi=200)
plt.show()


## How to interpret the results

1. **RQ1:** Rank AraBERT, MARBERT, and QARiB by the **overall Macro-F1** table.
2. **RQ2:** Use the per-aspect table to identify which model performs best for each aspect and which aspects are hardest.
3. Do **not** call a difference “statistically significant” unless you run a significance test.
4. Discuss class imbalance explicitly. In particular, the heritage subset has extremely few neutral examples for cleanliness and service/staff.
5. Compare your actual findings with Zaid et al. (2025), but do not force your result to match their full-tourism-dataset result.
6. Use `best_model_test_predictions.csv` as the input to the prototype dashboard.
